# LLM Text-Only Smoke Test

Goal: verify the LLM client uses plain text prompts and returns plain text responses.

This notebook includes:
1) An offline stub test with no network.
2) A live test that calls your configured LLM with plain text only.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

from dotenv import load_dotenv


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "fresh_simple_trading_project").exists():
            return candidate
        if (candidate / "fresh_simple_trading_project" / "src" / "fresh_simple_trading_project").exists():
            return candidate / "fresh_simple_trading_project"
    raise RuntimeError("Could not find the fresh_simple_trading_project root from the current working directory.")


project_root = find_project_root()
sys.path.insert(0, str(project_root / "src"))
load_dotenv(project_root.parent / ".env")
load_dotenv(project_root / ".env")

print("project_root=", project_root)


## 1) Offline smoke test

This validates the text-only `generate(...)` path with a stubbed streamed response.


In [ ]:
from types import SimpleNamespace

from fresh_simple_trading_project.config import LLMConfig
from fresh_simple_trading_project.llm import DeepSeekLLMClient


def _fake_stream(text: str):
    return [SimpleNamespace(choices=[SimpleNamespace(delta=SimpleNamespace(content=text))])]


offline_client = DeepSeekLLMClient(LLMConfig(api_key="test-key", show_progress=False))
offline_client._client = SimpleNamespace(
    chat=SimpleNamespace(
        completions=SimpleNamespace(
            create=lambda **_: _fake_stream("Plain text summary from the offline smoke test.")
        )
    )
)

offline_reply = offline_client.generate(
    "You are a concise trading analyst.",
    "Symbol: AAPL\nTrend: uptrend\nReturn one short summary.",
)
print(offline_reply)


## 2) Live smoke test

Requirements:
- Set `DEEPSEEK_API_KEY` and optionally `DEEPSEEK_MODEL` / `DEEPSEEK_BASE_URL` in `.env`.
- The prompt below is plain text only.


In [ ]:
from fresh_simple_trading_project.config import Settings
from fresh_simple_trading_project.llm import DeepSeekLLMClient

settings = Settings.from_env(project_root=project_root)
settings.llm.require()

llm = DeepSeekLLMClient(settings.llm)
symbol = settings.trading.symbol

system_prompt = "You are the news research agent in a multi-agent trading workflow. Review the provided news context and return one concise handoff."
user_prompt = "\n".join(
    [
        f"Symbol: {symbol}",
        "Search focus: symbol-specific news, American economy developments, and overall U.S. stock market trend.",
        "Queries used:",
        f"- latest {symbol} earnings news",
        f"- latest {symbol} product launch news",
        "Rule-based sentiment score: 0.35",
        "Risk flags: ['macro volatility']",
        "Catalysts: ['earnings beat', 'guidance raised']",
        "Recent articles:",
        f"- {symbol} beats expectations | Revenue and margins improved year over year. | source=demo | published_at=2025-01-03T10:00:00Z",
        f"- {symbol} guides higher | Management raised full-year outlook. | source=demo | published_at=2025-01-03T09:30:00Z",
        "Return one short news handoff for the decision-making agent.",
    ]
)

live_reply = llm.generate(system_prompt, user_prompt)
print(live_reply)
